In [1]:
# ============================================================
# FINAL ORIGINAL RUHSOLD XLM-R BASELINE
# BLOCK 1 - IMPORTS, PATHS, CONFIGURATION, REPRODUCIBILITY
#
# CANONICAL REFERENCE CONFIGURATION
#
# This baseline defines the XLM-R training configuration that
# must also be used for the unfiltered and QC-filtered runs.
#
# Across the three XLM-R experiments, the intended experimental
# difference is the TRAINING DATA only.
# ============================================================

from pathlib import Path

import os
import gc
import math
import random
import time
import shutil

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/project work/data_analyssis"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_train.tsv"
)

VAL_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_validation.tsv"
)

TEST_PATH = (
    PROJECT_ROOT
    / "RUHSOLD_test.tsv"
)


# ============================================================
# FINAL ORIGINAL BASELINE OUTPUT DIRECTORIES
# ============================================================

ORIGINAL_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "xlm_roberta_original_final_3seed"
)

ORIGINAL_RESULTS_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "xlm_roberta_original_final_3seed_results"
)


ORIGINAL_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ORIGINAL_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_NAME = (
    "FacebookAI/xlm-roberta-base"
)

NUM_LABELS = 5

MAX_LENGTH = 128


# ============================================================
# RUHSOLD LABEL MAPPING
# ============================================================

id2label = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane",
}

label2id = {
    label: class_id
    for class_id, label
    in id2label.items()
}


# ============================================================
# FROZEN OPTUNA-SELECTED HYPERPARAMETERS
#
# IMPORTANT:
# These values must remain unchanged across:
#
# 1. Original RUHSOLD XLM-R
# 2. Unfiltered 1x augmented XLM-R
# 3. QC-filtered 1x augmented XLM-R
#
# No hyperparameter retuning is performed for augmentation.
# ============================================================

BEST_LEARNING_RATE = (
    2.881129057462248e-05
)

BEST_WEIGHT_DECAY = (
    0.03348739395854274
)

BEST_WARMUP_RATIO = (
    0.03756172525242821
)

FINAL_MAX_EPOCHS = 10

FINAL_EARLY_STOPPING_PATIENCE = 2


# ============================================================
# CANONICAL BATCH CONFIGURATION
#
# This is now explicitly fixed so that the same configuration
# can be reused in the augmented experiments.
#
# Original successful baseline:
# physical train batch = 16
# gradient accumulation = 1
# effective train batch = 16
# evaluation batch      = 16
# ============================================================

TRAIN_BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 1

EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

EVAL_BATCH_SIZE = 16


assert (
    EFFECTIVE_BATCH_SIZE
    == 16
)


# Backward-compatible name used by earlier notebook code.
BEST_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
)


# ============================================================
# FINAL REPEATED-RUN SEEDS
# ============================================================

SEEDS = [
    42,
    43,
    44,
]


# ============================================================
# EXPECTED DATASET SIZES
# ============================================================

ORIGINAL_TRAIN_SIZE = 6408

ORIGINAL_VAL_SIZE = 801

ORIGINAL_TEST_SIZE = 2003


# ============================================================
# WARMUP CONVERSION
#
# The Optuna-selected warmup parameter is a RATIO.
#
# Because the installed Transformers version does not accept
# warmup_ratio, it is converted to the corresponding number
# of optimizer warmup steps.
#
# Original dataset:
#
# 6408 / physical batch 16
# = 401 mini-batches per epoch
#
# gradient accumulation = 1
#
# optimizer steps per epoch
# = 401
#
# 401 x 10 epochs
# = 4010 maximum optimizer steps
#
# ceil(
#     4010 x 0.03756172525242821
# )
# = 151 warmup steps
# ============================================================

MINI_BATCHES_PER_EPOCH = math.ceil(
    ORIGINAL_TRAIN_SIZE
    / TRAIN_BATCH_SIZE
)


STEPS_PER_EPOCH = math.ceil(
    MINI_BATCHES_PER_EPOCH
    / GRADIENT_ACCUMULATION_STEPS
)


MAX_TRAINING_STEPS = (
    STEPS_PER_EPOCH
    * FINAL_MAX_EPOCHS
)


BEST_WARMUP_STEPS = math.ceil(
    MAX_TRAINING_STEPS
    * BEST_WARMUP_RATIO
)


assert (
    MINI_BATCHES_PER_EPOCH
    == 401
)

assert (
    STEPS_PER_EPOCH
    == 401
)

assert (
    MAX_TRAINING_STEPS
    == 4010
)

assert (
    BEST_WARMUP_STEPS
    == 151
)


# ============================================================
# BASE REPRODUCIBILITY SETUP
# ============================================================

GLOBAL_SEED = 42


random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)

torch.manual_seed(
    GLOBAL_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        GLOBAL_SEED
    )


# ============================================================
# ENVIRONMENT + CONFIGURATION CHECK
# ============================================================

print("=" * 70)
print(
    "FINAL ORIGINAL RUHSOLD XLM-R BASELINE"
)
print("=" * 70)


print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


print(
    "\nModel:",
    MODEL_NAME
)


print(
    "Seeds:",
    SEEDS
)


print(
    "\nExpected original train:",
    ORIGINAL_TRAIN_SIZE
)

print(
    "Expected validation:",
    ORIGINAL_VAL_SIZE
)

print(
    "Expected test:",
    ORIGINAL_TEST_SIZE
)


print(
    "\nFrozen hyperparameters:"
)

print(
    "Learning rate:",
    BEST_LEARNING_RATE
)

print(
    "Weight decay:",
    BEST_WEIGHT_DECAY
)

print(
    "Frozen warmup ratio:",
    BEST_WARMUP_RATIO
)


print(
    "\nBatch configuration:"
)

print(
    "Physical training batch:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective training batch:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch:",
    EVAL_BATCH_SIZE
)


print(
    "\nTraining schedule:"
)

print(
    "Mini-batches per epoch:",
    MINI_BATCHES_PER_EPOCH
)

print(
    "Optimizer steps per epoch:",
    STEPS_PER_EPOCH
)

print(
    "Maximum optimizer steps:",
    MAX_TRAINING_STEPS
)

print(
    "Equivalent warmup steps:",
    BEST_WARMUP_STEPS
)

print(
    "Maximum epochs:",
    FINAL_MAX_EPOCHS
)

print(
    "Early stopping patience:",
    FINAL_EARLY_STOPPING_PATIENCE
)


print(
    "\nOutput root:"
)

print(
    ORIGINAL_OUTPUT_ROOT
)


print(
    "\nResults root:"
)

print(
    ORIGINAL_RESULTS_ROOT
)


print(
    "\nCANONICAL ORIGINAL XLM-R CONFIGURATION VERIFIED."
)

[HAMI-core Msg(156:140419248471360:libvgpu.c:839)]: Initializing.....
[HAMI-core Msg(156:140419248471360:libvgpu.c:855)]: Initialized


FINAL ORIGINAL RUHSOLD XLM-R BASELINE
CUDA available: True
GPU: NVIDIA L40S

Model: FacebookAI/xlm-roberta-base
Seeds: [42, 43, 44]

Expected original train: 6408
Expected validation: 801
Expected test: 2003

Frozen hyperparameters:
Learning rate: 2.881129057462248e-05
Weight decay: 0.03348739395854274
Frozen warmup ratio: 0.03756172525242821

Batch configuration:
Physical training batch: 16
Gradient accumulation: 1
Effective training batch: 16
Evaluation batch: 16

Training schedule:
Mini-batches per epoch: 401
Optimizer steps per epoch: 401
Maximum optimizer steps: 4010
Equivalent warmup steps: 151
Maximum epochs: 10
Early stopping patience: 2

Output root:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed

Results root:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed_results

CANONICAL ORIGINAL XLM-R CONFIGURATION VERIFIED.


In [2]:
# ============================================================
# FINAL ORIGINAL RUHSOLD XLM-R BASELINE
# BLOCK 2 - LOAD + VERIFY TRAIN / VALIDATION DATA
#
# CANONICAL DATA-INTEGRITY CHECK
# ============================================================

import re
import unicodedata
import pandas as pd


# ============================================================
# LOAD HEADERLESS RUHSOLD FILES
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    header=None,
    names=[
        "tweet",
        "label",
    ],
)

val_df = pd.read_csv(
    VAL_PATH,
    sep="\t",
    header=None,
    names=[
        "tweet",
        "label",
    ],
)


# ============================================================
# BASIC TYPE CLEANUP
# ============================================================

train_df["tweet"] = (
    train_df["tweet"]
    .astype(str)
)

val_df["tweet"] = (
    val_df["tweet"]
    .astype(str)
)


train_df["label"] = (
    train_df["label"]
    .astype(int)
)

val_df["label"] = (
    val_df["label"]
    .astype(int)
)


# ============================================================
# BASIC INTEGRITY CHECKS
# ============================================================

assert (
    len(train_df)
    ==
    ORIGINAL_TRAIN_SIZE
)

assert (
    len(val_df)
    ==
    ORIGINAL_VAL_SIZE
)


assert (
    train_df["tweet"]
    .isna()
    .sum()
    ==
    0
)

assert (
    train_df["label"]
    .isna()
    .sum()
    ==
    0
)


assert (
    val_df["tweet"]
    .isna()
    .sum()
    ==
    0
)

assert (
    val_df["label"]
    .isna()
    .sum()
    ==
    0
)


# ============================================================
# VALID LABEL SET
# ============================================================

VALID_LABELS = {
    0,
    1,
    2,
    3,
    4,
}


assert (
    set(
        train_df["label"]
        .unique()
    )
    ==
    VALID_LABELS
)


assert (
    set(
        val_df["label"]
        .unique()
    )
    ==
    VALID_LABELS
)


# ============================================================
# EXPECTED CLASS DISTRIBUTIONS
# ============================================================

EXPECTED_TRAIN_COUNTS = {
    0: 1537,
    1: 3423,
    2: 500,
    3: 537,
    4: 411,
}

EXPECTED_VAL_COUNTS = {
    0: 192,
    1: 428,
    2: 63,
    3: 67,
    4: 51,
}


actual_train_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
)


actual_val_counts = (
    val_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
)


assert (
    actual_train_counts
    ==
    EXPECTED_TRAIN_COUNTS
), (
    f"Unexpected train distribution.\n"
    f"Expected: {EXPECTED_TRAIN_COUNTS}\n"
    f"Actual:   {actual_train_counts}"
)


assert (
    actual_val_counts
    ==
    EXPECTED_VAL_COUNTS
), (
    f"Unexpected validation distribution.\n"
    f"Expected: {EXPECTED_VAL_COUNTS}\n"
    f"Actual:   {actual_val_counts}"
)


# ============================================================
# EXACT DUPLICATE CHECKS WITHIN EACH SPLIT
# ============================================================

train_exact_duplicates = (
    train_df
    .duplicated(
        subset=[
            "tweet",
            "label",
        ]
    )
    .sum()
)


val_exact_duplicates = (
    val_df
    .duplicated(
        subset=[
            "tweet",
            "label",
        ]
    )
    .sum()
)


print(
    "Exact duplicate train rows:",
    train_exact_duplicates
)

print(
    "Exact duplicate validation rows:",
    val_exact_duplicates
)


# ============================================================
# NORMALIZATION FUNCTION
#
# Used only for overlap auditing.
# Original tweet text itself is NOT modified for training.
# ============================================================

def normalize_text_for_overlap(text):

    text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    text = text.strip()

    return text


# ============================================================
# EXACT TRAIN / VALIDATION OVERLAP
# ============================================================

train_exact_texts = set(
    train_df["tweet"]
)

val_exact_texts = set(
    val_df["tweet"]
)


exact_train_val_overlap = (
    train_exact_texts
    &
    val_exact_texts
)


# ============================================================
# NORMALIZED TRAIN / VALIDATION OVERLAP
# ============================================================

train_normalized_texts = set(
    train_df["tweet"]
    .map(
        normalize_text_for_overlap
    )
)

val_normalized_texts = set(
    val_df["tweet"]
    .map(
        normalize_text_for_overlap
    )
)


normalized_train_val_overlap = (
    train_normalized_texts
    &
    val_normalized_texts
)


print(
    "\nExact train-validation text overlap:",
    len(
        exact_train_val_overlap
    )
)

print(
    "Normalized train-validation text overlap:",
    len(
        normalized_train_val_overlap
    )
)


# ============================================================
# DISPLAY VERIFIED DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print(
    "ORIGINAL RUHSOLD DATA VERIFICATION"
)
print("=" * 70)


print(
    "Train samples:",
    len(train_df)
)

print(
    "Validation samples:",
    len(val_df)
)


print(
    "\nMissing train tweets:",
    train_df["tweet"]
    .isna()
    .sum()
)

print(
    "Missing validation tweets:",
    val_df["tweet"]
    .isna()
    .sum()
)


print(
    "\nTraining class distribution:"
)

display(
    train_df["label"]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nValidation class distribution:"
)

display(
    val_df["label"]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


# ============================================================
# QUICK PREVIEW
# ============================================================

print(
    "\nTrain preview:"
)

display(
    train_df.head()
)


print(
    "\nValidation preview:"
)

display(
    val_df.head()
)


# ============================================================
# FINAL DATA-INTEGRITY SUMMARY
# ============================================================

print("\n" + "=" * 70)
print(
    "ORIGINAL RUHSOLD DATA-INTEGRITY SUMMARY"
)
print("=" * 70)


print(
    "Train size:",
    len(train_df)
)

print(
    "Validation size:",
    len(val_df)
)

print(
    "Train labels:",
    actual_train_counts
)

print(
    "Validation labels:",
    actual_val_counts
)

print(
    "Exact train-validation overlap:",
    len(
        exact_train_val_overlap
    )
)

print(
    "Normalized train-validation overlap:",
    len(
        normalized_train_val_overlap
    )
)


print(
    "\nORIGINAL RUHSOLD TRAIN / VALIDATION DATA VERIFIED."
)

Exact duplicate train rows: 6
Exact duplicate validation rows: 0

Exact train-validation text overlap: 0
Normalized train-validation text overlap: 0

ORIGINAL RUHSOLD DATA VERIFICATION
Train samples: 6408
Validation samples: 801

Missing train tweets: 0
Missing validation tweets: 0

Training class distribution:


,label,count
0,0,1537
1,1,3423
2,2,500
3,3,537
4,4,411



Validation class distribution:


,label,count
0,0,192
1,1,428
2,2,63
3,3,67
4,4,51



Train preview:


,tweet,label
0,kia howa hai aap ko allah bless and protect yo...,1
1,randdi hai,3
2,smjh to agai thi mjhy,1
3,haan yrr tuny sahi thukayi ki abhi tak lund da...,3
4,rundi ka bacha bharwaaa ptm ka kuttaa,0



Validation preview:


,tweet,label
0,lahore ki chae or coffee se bi masa hai inko d...,3
1,rt : 15 minutes kya 30 minutes bhaunk sakta ha...,0
2,ustad ye wala scene yes kraoo,1
3,bc kya maha fuddu banda hai ye,0
4,'crazy foodie' 😂😂😂😂 itna sach,1



ORIGINAL RUHSOLD DATA-INTEGRITY SUMMARY
Train size: 6408
Validation size: 801
Train labels: {0: 1537, 1: 3423, 2: 500, 3: 537, 4: 411}
Validation labels: {0: 192, 1: 428, 2: 63, 3: 67, 4: 51}
Exact train-validation overlap: 0
Normalized train-validation overlap: 0

ORIGINAL RUHSOLD TRAIN / VALIDATION DATA VERIFIED.


In [3]:
# ============================================================
# FINAL ORIGINAL RUHSOLD XLM-R BASELINE
# BLOCK 3 - TOKENIZER + DATASET + DATA COLLATOR
#
# CANONICAL TOKENIZATION / DATASET CONFIGURATION
#
# This exact tokenizer, max length, dataset structure, and
# dynamic-padding strategy must also be reused for the
# unfiltered and QC-filtered XLM-R experiments.
# ============================================================

from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
)


# ============================================================
# LOAD XLM-R TOKENIZER
#
# IMPORTANT:
# - Same tokenizer for all XLM-R experiments
# - No training-text normalization is applied here
# - MAX_LENGTH remains fixed at 128
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


print("=" * 70)
print("XLM-R TOKENIZER")
print("=" * 70)

print(
    "Tokenizer:",
    MODEL_NAME
)

print(
    "Vocabulary size:",
    tokenizer.vocab_size
)

print(
    "Tokenizer model maximum length:",
    tokenizer.model_max_length
)

print(
    "Experiment maximum length:",
    MAX_LENGTH
)

print(
    "Fast tokenizer:",
    tokenizer.is_fast
)


assert (
    tokenizer.is_fast
    is True
)

assert (
    MAX_LENGTH
    == 128
)


# ============================================================
# CUSTOM RUHSOLD DATASET
#
# Tokenization is performed independently for each sample.
#
# IMPORTANT:
# - truncation=True
# - max_length=128
# - padding=False here
#
# Padding is performed dynamically at batch level by
# DataCollatorWithPadding.
#
# This dataset class must be reused unchanged in all three
# XLM-R conditions.
# ============================================================

class RUHSOLDDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length,
    ):

        self.texts = (
            dataframe["tweet"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["label"]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer

        self.max_length = max_length


    def __len__(self):

        return len(
            self.labels
        )


    def __getitem__(
        self,
        idx,
    ):

        encoding = self.tokenizer(
            self.texts[idx],

            truncation=True,

            max_length=(
                self.max_length
            ),

            padding=False,
        )

        encoding["labels"] = (
            self.labels[idx]
        )

        return encoding


# ============================================================
# CREATE ORIGINAL TRAIN / VALIDATION DATASETS
# ============================================================

train_dataset = RUHSOLDDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = RUHSOLDDataset(
    dataframe=val_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)


# ============================================================
# DYNAMIC PADDING COLLATOR
#
# Same collator must be used for:
# - original baseline
# - unfiltered augmentation
# - QC-filtered augmentation
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
)


# ============================================================
# VERIFY DATASET SIZES
# ============================================================

assert (
    len(train_dataset)
    ==
    ORIGINAL_TRAIN_SIZE
)

assert (
    len(val_dataset)
    ==
    ORIGINAL_VAL_SIZE
)


# ============================================================
# VERIFY FIRST ITEMS
# ============================================================

first_train_item = (
    train_dataset[0]
)

first_val_item = (
    val_dataset[0]
)


required_keys = {
    "input_ids",
    "attention_mask",
    "labels",
}


assert required_keys.issubset(
    first_train_item.keys()
)

assert required_keys.issubset(
    first_val_item.keys()
)


assert (
    int(
        first_train_item["labels"]
    )
    ==
    int(
        train_df.iloc[0]["label"]
    )
)

assert (
    int(
        first_val_item["labels"]
    )
    ==
    int(
        val_df.iloc[0]["label"]
    )
)


# ============================================================
# VERIFY SEQUENCE LENGTH LIMIT
# ============================================================

assert (
    len(
        first_train_item[
            "input_ids"
        ]
    )
    <=
    MAX_LENGTH
)

assert (
    len(
        first_val_item[
            "input_ids"
        ]
    )
    <=
    MAX_LENGTH
)


# ============================================================
# VERIFY LABEL DISTRIBUTIONS INSIDE DATASET OBJECTS
# ============================================================

train_dataset_labels = np.asarray(
    train_dataset.labels,
    dtype=int,
)

val_dataset_labels = np.asarray(
    val_dataset.labels,
    dtype=int,
)


train_dataset_counts = {
    int(label): int(count)

    for label, count

    in zip(
        *np.unique(
            train_dataset_labels,
            return_counts=True,
        )
    )
}


val_dataset_counts = {
    int(label): int(count)

    for label, count

    in zip(
        *np.unique(
            val_dataset_labels,
            return_counts=True,
        )
    )
}


assert (
    train_dataset_counts
    ==
    EXPECTED_TRAIN_COUNTS
)

assert (
    val_dataset_counts
    ==
    EXPECTED_VAL_COUNTS
)


# ============================================================
# VERIFY COLLATOR ON A SMALL BATCH
# ============================================================

sample_batch = data_collator(
    [
        train_dataset[0],
        train_dataset[1],
        train_dataset[2],
        train_dataset[3],
    ]
)


assert (
    sample_batch[
        "input_ids"
    ].shape[0]
    ==
    4
)

assert (
    sample_batch[
        "attention_mask"
    ].shape[0]
    ==
    4
)

assert (
    sample_batch[
        "labels"
    ].shape[0]
    ==
    4
)


# Batch sequence length should never exceed MAX_LENGTH.
assert (
    sample_batch[
        "input_ids"
    ].shape[1]
    <=
    MAX_LENGTH
)


# ============================================================
# DISPLAY VERIFICATION
# ============================================================

print("\n" + "=" * 70)

print(
    "TOKENIZED DATASET VERIFICATION"
)

print("=" * 70)


print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)


print(
    "\nTraining class distribution:",
    train_dataset_counts
)

print(
    "Validation class distribution:",
    val_dataset_counts
)


print(
    "\nFirst training item keys:",
    first_train_item.keys()
)

print(
    "First training token length:",
    len(
        first_train_item[
            "input_ids"
        ]
    )
)

print(
    "First training label:",
    first_train_item[
        "labels"
    ]
)


print(
    "\nFirst validation item keys:",
    first_val_item.keys()
)

print(
    "First validation token length:",
    len(
        first_val_item[
            "input_ids"
        ]
    )
)

print(
    "First validation label:",
    first_val_item[
        "labels"
    ]
)


print(
    "\nTest collated batch shape:",
    sample_batch[
        "input_ids"
    ].shape
)

print(
    "Test collated label shape:",
    sample_batch[
        "labels"
    ].shape
)


print(
    "\nTOKENIZER / DATASET / COLLATOR "
    "CONFIGURATION VERIFIED."
)

XLM-R TOKENIZER
Tokenizer: FacebookAI/xlm-roberta-base
Vocabulary size: 250002
Tokenizer model maximum length: 512
Experiment maximum length: 128
Fast tokenizer: True

TOKENIZED DATASET VERIFICATION
Training samples: 6408
Validation samples: 801

Training class distribution: {0: 1537, 1: 3423, 2: 500, 3: 537, 4: 411}
Validation class distribution: {0: 192, 1: 428, 2: 63, 3: 67, 4: 51}

First training item keys: KeysView({'input_ids': [0, 19176, 3642, 11, 1337, 38674, 298, 6, 40077, 77805, 136, 59959, 398, 10, 15516, 33, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 1})
First training token length: 17
First training label: 1

First validation item keys: KeysView({'input_ids': [0, 21, 94187, 200, 1608, 13, 707, 79497, 40, 333, 3262, 1337, 23, 265, 48, 1021, 741, 79, 31, 1096, 1727, 53, 2412, 3467, 40, 6, 94138, 6456, 7, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 3})
Fir

In [7]:
======================================================================
SEED 42 DIAGNOSTIC CHECKPOINT STATUS
======================================================================
Output directory: /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42
Existing checkpoints: 1
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42/checkpoint-2005
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[5], line 113
    101 FORCE_RERUN_DIAGNOSTIC = False
    104 if (
    105     len(
    106         existing_diagnostic_checkpoints
   (...)    110     not FORCE_RERUN_DIAGNOSTIC
    111 ):
--> 113     raise RuntimeError(
    114         "\nA seed-42 diagnostic checkpoint already exists.\n"
    115         "It has NOT been deleted.\n\n"
    116         "If this is the valid completed seed-42 run, "
    117         "keep it and do not rerun Block 5.\n\n"
    118         "Only set FORCE_RERUN_DIAGNOSTIC=True if you "
    119         "deliberately want to replace that checkpoint."
    120     )
    123 # ============================================================
    124 # REMOVE OLD RUN ONLY WHEN EXPLICITLY REQUESTED
    125 # ============================================================
    127 if (
    128     FORCE_RERUN_DIAGNOSTIC
    129     and
    130     DIAGNOSTIC_OUTPUT_DIR.exists()
    131 ):

RuntimeError: 
A seed-42 diagnostic checkpoint already exists.
It has NOT been deleted.

If this is the valid completed seed-42 run, keep it and do not rerun Block 5.

Only set FORCE_RERUN_DIAGNOSTIC=True if you deliberately want to replace that checkpoint

SyntaxError: invalid syntax (4193148544.py, line 1)

In [8]:
# ============================================================
# FINAL ORIGINAL RUHSOLD XLM-R BASELINE
# BLOCK 5 - SEED 42 CANONICAL DIAGNOSTIC TRAINING
#
# PURPOSE:
# - Establish the canonical XLM-R training implementation
# - Original RUHSOLD training set only
# - Same original validation set
# - Same frozen Optuna hyperparameters
# - Model loaded normally in FP32
# - BF16 used ONLY as Trainer mixed precision
# - Validation Macro-F1 selects the best checkpoint
#
# THIS MODEL-LOADING / TRAINING-PRECISION SETUP MUST LATER BE
# COPIED TO BOTH AUGMENTED PIPELINES.
# ============================================================

import gc
import time
import shutil

from pathlib import Path

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)


# ============================================================
# DIAGNOSTIC SEED
# ============================================================

DIAGNOSTIC_SEED = 42


# ============================================================
# DIAGNOSTIC OUTPUT DIRECTORY
# ============================================================

DIAGNOSTIC_OUTPUT_DIR = (
    ORIGINAL_OUTPUT_ROOT
    / "diagnostic_seed_42"
)


# ============================================================
# CHECK FOR EXISTING VALID CHECKPOINT
#
# IMPORTANT:
# Do NOT automatically delete a completed dissertation run.
# ============================================================

existing_diagnostic_checkpoints = []

if DIAGNOSTIC_OUTPUT_DIR.exists():

    existing_diagnostic_checkpoints = sorted(
        DIAGNOSTIC_OUTPUT_DIR.glob(
            "checkpoint-*"
        )
    )


print("=" * 70)
print("SEED 42 DIAGNOSTIC CHECKPOINT STATUS")
print("=" * 70)

print(
    "Output directory:",
    DIAGNOSTIC_OUTPUT_DIR
)

print(
    "Existing checkpoints:",
    len(
        existing_diagnostic_checkpoints
    )
)

for checkpoint in (
    existing_diagnostic_checkpoints
):

    print(
        checkpoint
    )


# ============================================================
# SAFETY SWITCH
#
# Leave False unless you intentionally want to retrain seed 42.
#
# Your already-completed valid seed-42 checkpoint should
# normally be preserved.
# ============================================================

FORCE_RERUN_DIAGNOSTIC = False


if (
    len(
        existing_diagnostic_checkpoints
    )
    > 0
    and
    not FORCE_RERUN_DIAGNOSTIC
):

    raise RuntimeError(
        "\nA seed-42 diagnostic checkpoint already exists.\n"
        "It has NOT been deleted.\n\n"
        "If this is the valid completed seed-42 run, "
        "keep it and do not rerun Block 5.\n\n"
        "Only set FORCE_RERUN_DIAGNOSTIC=True if you "
        "deliberately want to replace that checkpoint."
    )


# ============================================================
# REMOVE OLD RUN ONLY WHEN EXPLICITLY REQUESTED
# ============================================================

if (
    FORCE_RERUN_DIAGNOSTIC
    and
    DIAGNOSTIC_OUTPUT_DIR.exists()
):

    print(
        "\nExplicit rerun requested."
    )

    print(
        "Removing old diagnostic directory:"
    )

    print(
        DIAGNOSTIC_OUTPUT_DIR
    )

    shutil.rmtree(
        DIAGNOSTIC_OUTPUT_DIR
    )


DIAGNOSTIC_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

set_seed(
    DIAGNOSTIC_SEED
)


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# LOAD FRESH PRETRAINED XLM-R
#
# CRITICAL ISSUE-1 FIX:
#
# DO NOT use:
#
# dtype=torch.bfloat16
#
# The pretrained parameters must first be loaded normally.
#
# Expected parameter dtype before Trainer:
# torch.float32
#
# BF16 is enabled only through TrainingArguments below.
# ============================================================

model = (
    AutoModelForSequenceClassification
    .from_pretrained(

        MODEL_NAME,

        num_labels=NUM_LABELS,

        id2label=id2label,

        label2id=label2id,
    )
)


model_parameter_dtype = (
    next(
        model.parameters()
    ).dtype
)


print("\n" + "=" * 70)
print("SEED 42 CANONICAL MODEL CHECK")
print("=" * 70)

print(
    "Model:",
    MODEL_NAME
)

print(
    "Model parameter dtype before Trainer:",
    model_parameter_dtype
)


# ============================================================
# CRITICAL PRECISION ASSERTION
# ============================================================

assert (
    model_parameter_dtype
    ==
    torch.float32
), (
    "XLM-R was not loaded in FP32. "
    "Do not use dtype=torch.bfloat16 in from_pretrained()."
)


print(
    "FP32 pretrained model loading: VERIFIED"
)


# ============================================================
# TRAINING CONFIGURATION CHECK
# ============================================================

print(
    "\nCanonical batch configuration:"
)

print(
    "Physical train batch:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective train batch:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch:",
    EVAL_BATCH_SIZE
)

print(
    "Warmup steps:",
    BEST_WARMUP_STEPS
)


assert (
    TRAIN_BATCH_SIZE
    == 16
)

assert (
    GRADIENT_ACCUMULATION_STEPS
    == 1
)

assert (
    EFFECTIVE_BATCH_SIZE
    == 16
)

assert (
    EVAL_BATCH_SIZE
    == 16
)

assert (
    BEST_WARMUP_STEPS
    == 151
)


# ============================================================
# TRAINING ARGUMENTS
# ============================================================

diagnostic_training_args = TrainingArguments(

    output_dir=str(
        DIAGNOSTIC_OUTPUT_DIR
    ),

    # --------------------------------------------------------
    # Evaluation / logging
    # --------------------------------------------------------

    eval_strategy="epoch",

    logging_strategy="epoch",

    # --------------------------------------------------------
    # Best-checkpoint saving
    # --------------------------------------------------------

    save_strategy="best",

    save_total_limit=1,

    save_only_model=True,

    # --------------------------------------------------------
    # FROZEN OPTUNA HYPERPARAMETERS
    # --------------------------------------------------------

    learning_rate=(
        BEST_LEARNING_RATE
    ),

    weight_decay=(
        BEST_WEIGHT_DECAY
    ),

    # --------------------------------------------------------
    # CANONICAL BATCH CONFIGURATION
    # --------------------------------------------------------

    per_device_train_batch_size=(
        TRAIN_BATCH_SIZE
    ),

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    per_device_eval_batch_size=(
        EVAL_BATCH_SIZE
    ),

    # --------------------------------------------------------
    # SAME FROZEN WARMUP RATIO
    #
    # Original-data equivalent = 151 optimizer steps.
    # --------------------------------------------------------

    warmup_steps=(
        BEST_WARMUP_STEPS
    ),

    # --------------------------------------------------------
    # TRAINING DURATION
    # --------------------------------------------------------

    num_train_epochs=(
        FINAL_MAX_EPOCHS
    ),

    # --------------------------------------------------------
    # VALIDATION-ONLY CHECKPOINT SELECTION
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model=(
        CHECKPOINT_SELECTION_METRIC
    ),

    greater_is_better=True,

    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------

    seed=(
        DIAGNOSTIC_SEED
    ),

    data_seed=(
        DIAGNOSTIC_SEED
    ),

    # --------------------------------------------------------
    # MIXED PRECISION
    #
    # IMPORTANT:
    #
    # The MODEL was loaded in FP32.
    # BF16 is enabled only for mixed-precision training.
    # --------------------------------------------------------

    bf16=(
        torch.cuda.is_available()
    ),

    fp16=False,

    # --------------------------------------------------------
    # MISC
    # --------------------------------------------------------

    report_to="none",

    disable_tqdm=False,
)


# ============================================================
# TRAINER
# ============================================================

diagnostic_trainer = Trainer(

    model=model,

    args=(
        diagnostic_training_args
    ),

    train_dataset=(
        train_dataset
    ),

    eval_dataset=(
        val_dataset
    ),

    data_collator=(
        data_collator
    ),

    compute_metrics=(
        compute_metrics
    ),

    processing_class=(
        tokenizer
    ),

    callbacks=[

        EarlyStoppingCallback(

            early_stopping_patience=(
                FINAL_EARLY_STOPPING_PATIENCE
            ),

            early_stopping_threshold=0.0,
        )
    ],
)


# ============================================================
# FINAL PRE-TRAINING AUDIT
# ============================================================

print("\n" + "=" * 80)
print("CANONICAL ORIGINAL XLM-R TRAINING AUDIT")
print("=" * 80)


print(
    "Training samples:",
    len(
        train_dataset
    )
)

print(
    "Validation samples:",
    len(
        val_dataset
    )
)

print(
    "Seed:",
    DIAGNOSTIC_SEED
)

print(
    "Model parameter dtype:",
    model_parameter_dtype
)

print(
    "Physical train batch:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch:",
    EVAL_BATCH_SIZE
)

print(
    "Learning rate:",
    BEST_LEARNING_RATE
)

print(
    "Weight decay:",
    BEST_WEIGHT_DECAY
)

print(
    "Warmup ratio:",
    BEST_WARMUP_RATIO
)

print(
    "Warmup steps:",
    BEST_WARMUP_STEPS
)

print(
    "Maximum epochs:",
    FINAL_MAX_EPOCHS
)

print(
    "Early stopping patience:",
    FINAL_EARLY_STOPPING_PATIENCE
)

print(
    "Checkpoint-selection metric:",
    CHECKPOINT_SELECTION_METRIC
)

print(
    "Trainer BF16 mixed precision:",
    diagnostic_training_args.bf16
)


# ============================================================
# TRAIN
# ============================================================

print("\n" + "=" * 80)

print(
    "ORIGINAL RUHSOLD XLM-R "
    "DIAGNOSTIC RUN - SEED 42"
)

print("=" * 80)


start_time = time.time()


diagnostic_trainer.train()


training_time = (
    time.time()
    - start_time
)


# ============================================================
# BEST CHECKPOINT INFORMATION
# ============================================================

best_checkpoint = (
    diagnostic_trainer
    .state
    .best_model_checkpoint
)


best_validation_macro_f1 = (
    diagnostic_trainer
    .state
    .best_metric
)


epoch_reached = (
    diagnostic_trainer
    .state
    .epoch
)


assert (
    best_checkpoint
    is not None
)


best_checkpoint_path = Path(
    best_checkpoint
)


assert (
    best_checkpoint_path.exists()
)


print(
    "\nBest checkpoint:"
)

print(
    best_checkpoint
)


print(
    "Best validation Macro F1:",
    f"{best_validation_macro_f1:.4f}"
)


print(
    "Epoch reached:",
    epoch_reached
)


print(
    "Training time:",
    f"{training_time / 60:.2f} minutes"
)


# ============================================================
# VALIDATION PREDICTIONS USING BEST MODEL
# ============================================================

prediction_output = (
    diagnostic_trainer.predict(
        val_dataset
    )
)


y_true = (
    prediction_output.label_ids
)


y_pred = np.argmax(
    prediction_output.predictions,
    axis=1,
)


assert (
    len(
        y_true
    )
    ==
    ORIGINAL_VAL_SIZE
)

assert (
    len(
        y_pred
    )
    ==
    ORIGINAL_VAL_SIZE
)


# ============================================================
# FULL CLASSIFICATION REPORT
# ============================================================

diagnostic_report = classification_report(

    y_true,

    y_pred,

    labels=(
        METRIC_LABELS
    ),

    target_names=[
        id2label[
            class_id
        ]
        for class_id
        in METRIC_LABELS
    ],

    output_dict=True,

    zero_division=0,
)


diagnostic_report_df = (
    pd.DataFrame(
        diagnostic_report
    )
    .transpose()
)


print(
    "\nSEED 42 DIAGNOSTIC VALIDATION REPORT:"
)


display(
    diagnostic_report_df.round(4)
)


# ============================================================
# CHECKPOINT RETENTION
# ============================================================

assert (
    best_checkpoint_path.exists()
)


print(
    "\nDiagnostic checkpoint retained:"
)

print(
    best_checkpoint_path
)


# ============================================================
# FINAL DIAGNOSTIC SUMMARY
# ============================================================

print("\n" + "=" * 80)

print(
    "SEED 42 CANONICAL DIAGNOSTIC COMPLETE"
)

print("=" * 80)


print(
    "Validation Macro F1:",
    f"{best_validation_macro_f1:.4f}"
)


print(
    "Model loaded initially as:",
    model_parameter_dtype
)


print(
    "Training mixed precision:",
    "BF16"
    if diagnostic_training_args.bf16
    else "FP32"
)


print(
    "\nCANONICAL PRECISION CONFIGURATION:"
)

print(
    "Pretrained parameters -> FP32"
)

print(
    "Trainer computation    -> BF16 mixed precision"
)


print(
    "\nThis exact model-loading and precision configuration "
    "must be reused in both augmented XLM-R experiments."
)

SEED 42 DIAGNOSTIC CHECKPOINT STATUS
Output directory: /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42
Existing checkpoints: 1
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42/checkpoint-2005


RuntimeError: 
A seed-42 diagnostic checkpoint already exists.
It has NOT been deleted.

If this is the valid completed seed-42 run, keep it and do not rerun Block 5.

Only set FORCE_RERUN_DIAGNOSTIC=True if you deliberately want to replace that checkpoint.

In [5]:
# ============================================================
# FINAL ORIGINAL RUHSOLD XLM-R BASELINE
# BLOCK 6 - TRAIN SEEDS 43 AND 44 ONLY
#
# IMPORTANT:
# - Seed 42 is already completed and retained.
# - This block trains ONLY seeds 43 and 44.
# - Same original RUHSOLD train/validation data.
# - Same frozen Optuna hyperparameters.
# - Model is loaded normally in FP32.
# - BF16 is enabled only through TrainingArguments.
# - Best checkpoint selected by validation Macro-F1.
# - Best checkpoint is RETAINED.
# ============================================================

import gc
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import classification_report

from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)


# ============================================================
# REMAINING SEEDS ONLY
# ============================================================

REMAINING_SEEDS = [
    43,
    44,
]


# ============================================================
# RESULT STORAGE FOR SEEDS 43 AND 44
# ============================================================

remaining_overall_results = []

remaining_per_class_results = []


# ============================================================
# TRAIN SEEDS 43 AND 44
# ============================================================

for seed in REMAINING_SEEDS:

    print("\n")
    print("=" * 80)

    print(
        f"ORIGINAL RUHSOLD XLM-R FINAL RUN - SEED {seed}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    set_seed(
        seed
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Seed-specific output directory
    # --------------------------------------------------------

    seed_output_dir = (
        ORIGINAL_OUTPUT_ROOT
        / f"seed_{seed}"
    )


    # --------------------------------------------------------
    # Clear only incomplete/old attempt for this seed
    # --------------------------------------------------------

    if seed_output_dir.exists():

        shutil.rmtree(
            seed_output_dir
        )


    seed_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # LOAD FRESH XLM-R
    #
    # CRITICAL:
    # NO dtype=torch.bfloat16 here.
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id,
        )
    )


    print(
        "Model parameter dtype before Trainer:",
        next(
            model.parameters()
        ).dtype
    )


    assert (
        next(
            model.parameters()
        ).dtype
        ==
        torch.float32
    )


    # ========================================================
    # TRAINING ARGUMENTS
    # ========================================================

    training_args = TrainingArguments(

        output_dir=str(
            seed_output_dir
        ),

        # ----------------------------------------------------
        # Evaluation / logging
        # ----------------------------------------------------

        eval_strategy="epoch",

        logging_strategy="epoch",

        # ----------------------------------------------------
        # Validation-selected best checkpoint
        # ----------------------------------------------------

        save_strategy="best",

        save_total_limit=1,

        save_only_model=True,

        # ----------------------------------------------------
        # Frozen Optuna-selected hyperparameters
        # ----------------------------------------------------

        learning_rate=(
            BEST_LEARNING_RATE
        ),

        per_device_train_batch_size=(
            BEST_BATCH_SIZE
        ),

        per_device_eval_batch_size=(
            BEST_BATCH_SIZE
        ),

        weight_decay=(
            BEST_WEIGHT_DECAY
        ),

        warmup_steps=(
            BEST_WARMUP_STEPS
        ),

        num_train_epochs=(
            FINAL_MAX_EPOCHS
        ),

        # ----------------------------------------------------
        # Best-model selection
        # ----------------------------------------------------

        load_best_model_at_end=True,

        metric_for_best_model=(
            "macro_f1"
        ),

        greater_is_better=True,

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------

        seed=seed,

        data_seed=seed,

        # ----------------------------------------------------
        # Mixed precision
        # ----------------------------------------------------

        bf16=torch.cuda.is_available(),

        fp16=False,

        report_to="none",

        disable_tqdm=False,
    )


    # ========================================================
    # TRAINER
    # ========================================================

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=(
            train_dataset
        ),

        eval_dataset=(
            val_dataset
        ),

        data_collator=(
            data_collator
        ),

        compute_metrics=(
            compute_metrics
        ),

        processing_class=(
            tokenizer
        ),

        callbacks=[

            EarlyStoppingCallback(

                early_stopping_patience=(
                    FINAL_EARLY_STOPPING_PATIENCE
                ),

                early_stopping_threshold=0.0,
            )
        ],
    )


    # ========================================================
    # TRAIN
    # ========================================================

    start_time = time.time()

    trainer.train()

    training_time = (
        time.time()
        - start_time
    )


    # ========================================================
    # BEST CHECKPOINT INFORMATION
    # ========================================================

    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )

    best_checkpoint_path = (
        Path(
            best_checkpoint
        )
        if best_checkpoint is not None
        else None
    )

    best_validation_macro_f1 = (
        trainer.state.best_metric
    )

    epoch_reached = (
        trainer.state.epoch
    )


    print(
        "\nBest checkpoint:"
    )

    print(
        best_checkpoint
    )


    print(
        "Best validation Macro F1:",
        f"{best_validation_macro_f1:.4f}"
    )


    print(
        "Epoch reached:",
        epoch_reached
    )


    print(
        "Training time:",
        f"{training_time / 60:.2f} minutes"
    )


    # ========================================================
    # VALIDATION PREDICTIONS
    # ========================================================

    prediction_output = (
        trainer.predict(
            val_dataset
        )
    )

    y_true = (
        prediction_output.label_ids
    )

    y_pred = np.argmax(
        prediction_output.predictions,
        axis=1,
    )


    assert len(y_true) == 801
    assert len(y_pred) == 801


    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    report = classification_report(

        y_true,

        y_pred,

        labels=[
            0,
            1,
            2,
            3,
            4,
        ],

        target_names=[
            "Abusive/Offensive",
            "Normal",
            "Religious Hate",
            "Sexism",
            "Profane",
        ],

        output_dict=True,

        zero_division=0,
    )


    report_df = (
        pd.DataFrame(
            report
        )
        .transpose()
    )


    print(
        f"\nPER-CLASS VALIDATION RESULTS - SEED {seed}"
    )


    display(
        report_df.round(4)
    )


    # ========================================================
    # STORE OVERALL RESULT
    # ========================================================

    remaining_overall_results.append({

        "seed":
            seed,

        "best_checkpoint":
            str(
                best_checkpoint
            ),

        "epoch_reached":
            epoch_reached,

        "best_validation_macro_f1":
            best_validation_macro_f1,

        "validation_accuracy":
            report[
                "accuracy"
            ],

        "validation_macro_precision":
            report[
                "macro avg"
            ][
                "precision"
            ],

        "validation_macro_recall":
            report[
                "macro avg"
            ][
                "recall"
            ],

        "validation_macro_f1":
            report[
                "macro avg"
            ][
                "f1-score"
            ],

        "validation_weighted_f1":
            report[
                "weighted avg"
            ][
                "f1-score"
            ],

        "training_time_minutes":
            training_time / 60,
    })


    # ========================================================
    # STORE PER-CLASS RESULTS
    # ========================================================

    for class_id in range(5):

        class_name = (
            id2label[
                class_id
            ]
        )

        class_metrics = (
            report[
                class_name
            ]
        )


        remaining_per_class_results.append({

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        })


    # ========================================================
    # SAVE VALIDATION PREDICTIONS
    # ========================================================

    prediction_df = pd.DataFrame({

        "true_label_id":
            y_true,

        "predicted_label_id":
            y_pred,

        "true_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_true
            ],

        "predicted_label":
            [
                id2label[
                    int(label)
                ]
                for label
                in y_pred
            ],
    })


    prediction_df.to_csv(

        ORIGINAL_RESULTS_ROOT
        / (
            f"seed_{seed}_"
            "validation_predictions.csv"
        ),

        index=False,
    )


    # ========================================================
    # SAVE RESULTS INCREMENTALLY
    # ========================================================

    pd.DataFrame(
        remaining_overall_results
    ).to_csv(

        ORIGINAL_RESULTS_ROOT
        / "remaining_seed_validation_results.csv",

        index=False,
    )


    pd.DataFrame(
        remaining_per_class_results
    ).to_csv(

        ORIGINAL_RESULTS_ROOT
        / "remaining_seed_per_class_results.csv",

        index=False,
    )


    # ========================================================
    # RETAIN BEST CHECKPOINT
    # ========================================================

    if (
        best_checkpoint_path is None
        or
        not best_checkpoint_path.exists()
    ):

        raise RuntimeError(
            f"Best checkpoint missing for seed {seed}"
        )


    print(
        "\nCheckpoint retained:"
    )

    print(
        best_checkpoint_path
    )


    # ========================================================
    # MEMORY CLEANUP ONLY
    # ========================================================

    del prediction_output
    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Verify checkpoint survived cleanup
    # --------------------------------------------------------

    assert (
        best_checkpoint_path.exists()
    )


    print(
        "Checkpoint confirmed after cleanup:"
    )

    print(
        best_checkpoint_path
    )


    # --------------------------------------------------------
    # Disk-space report
    # --------------------------------------------------------

    total, used, free = shutil.disk_usage(
        "/home/jovyan"
    )


    print(
        "Disk free:",
        f"{free / 1024**3:.2f} GB"
    )


# ============================================================
# FINAL CHECKPOINT COLLECTION
# ============================================================

print("\n")
print("=" * 80)

print(
    "FINAL ORIGINAL XLM-R CHECKPOINTS"
)

print("=" * 80)


# ------------------------------------------------------------
# Seed 42 diagnostic checkpoint
# ------------------------------------------------------------

SEED42_CHECKPOINT = (
    ORIGINAL_OUTPUT_ROOT
    / "diagnostic_seed_42"
    / "checkpoint-2005"
)


assert (
    SEED42_CHECKPOINT.exists()
)


print(
    "Seed 42:"
)

print(
    SEED42_CHECKPOINT
)


# ------------------------------------------------------------
# Seeds 43 and 44
# ------------------------------------------------------------

FINAL_ORIGINAL_CHECKPOINTS = {
    42: SEED42_CHECKPOINT,
}


for seed in [
    43,
    44,
]:

    seed_dir = (
        ORIGINAL_OUTPUT_ROOT
        / f"seed_{seed}"
    )


    checkpoints = sorted(
        seed_dir.glob(
            "checkpoint-*"
        )
    )


    assert (
        len(checkpoints)
        == 1
    ), (
        f"Expected exactly one retained checkpoint "
        f"for seed {seed}; found {len(checkpoints)}."
    )


    FINAL_ORIGINAL_CHECKPOINTS[
        seed
    ] = checkpoints[0]


    print(
        f"\nSeed {seed}:"
    )

    print(
        checkpoints[0]
    )


# ============================================================
# FINAL VERIFICATION
# ============================================================

assert set(
    FINAL_ORIGINAL_CHECKPOINTS.keys()
) == {
    42,
    43,
    44,
}


print("\n" + "=" * 80)

print(
    "ALL THREE ORIGINAL XLM-R CHECKPOINTS READY FOR TEST"
)

print("=" * 80)


for seed, checkpoint in (
    FINAL_ORIGINAL_CHECKPOINTS.items()
):

    print(
        f"Seed {seed}:",
        checkpoint
    )

SEED 42 DIAGNOSTIC CHECKPOINT STATUS
Output directory: /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42
Existing checkpoints: 1
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42/checkpoint-2005


RuntimeError: 
A seed-42 diagnostic checkpoint already exists.
It has NOT been deleted.

If this is the valid completed seed-42 run, keep it and do not rerun Block 5.

Only set FORCE_RERUN_DIAGNOSTIC=True if you deliberately want to replace that checkpoint.

In [10]:
# ============================================================
# FINAL ORIGINAL RUHSOLD XLM-R BASELINE
# BLOCK 7 - FINAL THREE-SEED TEST EVALUATION
#
# IMPORTANT:
# - NO TRAINING occurs here.
# - Uses ONLY the retained validation-selected checkpoints.
# - Test set is used ONLY for final evaluation.
# - Seeds 42, 43, 44 are evaluated independently.
# - Same test protocol as unfiltered and QC-filtered runs.
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. FINAL ORIGINAL CHECKPOINTS
# ============================================================

FINAL_ORIGINAL_CHECKPOINTS = {

    42: (
        ORIGINAL_OUTPUT_ROOT
        / "diagnostic_seed_42"
        / "checkpoint-2005"
    ),

    43: (
        ORIGINAL_OUTPUT_ROOT
        / "seed_43"
        / "checkpoint-2005"
    ),

    44: (
        ORIGINAL_OUTPUT_ROOT
        / "seed_44"
        / "checkpoint-1203"
    ),
}


# ============================================================
# 2. FINAL TEST RESULTS DIRECTORY
# ============================================================

FINAL_ORIGINAL_TEST_DIR = (
    ORIGINAL_RESULTS_ROOT
    / "final_test"
)

FINAL_ORIGINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. LABEL CONFIGURATION
# ============================================================

LABEL_IDS = [
    0,
    1,
    2,
    3,
    4,
]

LABEL_NAMES = [
    id2label[i]
    for i in LABEL_IDS
]

TEST_BATCH_SIZE = 4


# ============================================================
# 4. VERIFY ALL THREE CHECKPOINTS
# ============================================================

print("=" * 70)
print("ORIGINAL XLM-R FINAL TEST CHECKPOINTS")
print("=" * 70)


for seed in [
    42,
    43,
    44,
]:

    checkpoint_path = (
        FINAL_ORIGINAL_CHECKPOINTS[
            seed
        ]
    )

    print(
        f"Seed {seed}:",
        checkpoint_path.exists(),
        checkpoint_path,
    )

    assert checkpoint_path.exists(), (
        f"Missing checkpoint for seed {seed}: "
        f"{checkpoint_path}"
    )

    weight_files = (
        list(
            checkpoint_path.glob(
                "*.safetensors"
            )
        )
        +
        list(
            checkpoint_path.glob(
                "pytorch_model*.bin"
            )
        )
    )

    assert len(weight_files) > 0, (
        f"No model weights found for seed {seed}"
    )


print(
    "\nAll three original XLM-R checkpoints verified."
)


# ============================================================
# 5. LOAD UNTOUCHED RUHSOLD TEST SET
#
# File is headerless.
# ============================================================

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t",
    header=None,
    names=[
        "tweet",
        "label",
    ],
)


test_df["label"] = (
    test_df["label"]
    .astype(int)
)


# ============================================================
# 6. TEST-SET INTEGRITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL TEST-SET INTEGRITY CHECK")
print("=" * 70)

print(
    "Test samples:",
    len(test_df)
)

print(
    "Missing tweets:",
    test_df[
        "tweet"
    ]
    .isna()
    .sum()
)

print(
    "Missing labels:",
    test_df[
        "label"
    ]
    .isna()
    .sum()
)


assert len(test_df) == 2003

assert (
    test_df["tweet"]
    .isna()
    .sum()
    == 0
)

assert (
    test_df["label"]
    .isna()
    .sum()
    == 0
)


EXPECTED_TEST_COUNTS = {
    0: 481,
    1: 1070,
    2: 156,
    3: 168,
    4: 128,
}


actual_test_counts = (
    test_df["label"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print(
    "\nTest class distribution:"
)


display(
    test_df["label"]
    .value_counts()
    .sort_index()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="count"
    )
)


assert (
    actual_test_counts
    ==
    EXPECTED_TEST_COUNTS
), (
    "Unexpected test class distribution.\n"
    f"Expected: {EXPECTED_TEST_COUNTS}\n"
    f"Actual:   {actual_test_counts}"
)


print(
    "\nTest-set integrity check: PASSED"
)


# ============================================================
# 7. TOKENIZER
# ============================================================

test_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


def tokenize_test_function(batch):

    return test_tokenizer(
        batch["tweet"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


# ============================================================
# 8. CREATE FIXED TEST DATASET
# ============================================================

test_dataset_hf = Dataset.from_pandas(
    test_df[
        [
            "tweet",
            "label",
        ]
    ],
    preserve_index=False,
)


test_dataset_hf = (
    test_dataset_hf.map(
        tokenize_test_function,
        batched=True,
    )
)


test_dataset_hf = (
    test_dataset_hf.rename_column(
        "label",
        "labels",
    )
)


test_dataset_hf.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)


test_collator = (
    DataCollatorWithPadding(
        tokenizer=test_tokenizer
    )
)


# ============================================================
# 9. FIXED TEST DATALOADER
# ============================================================

test_loader = DataLoader(
    test_dataset_hf,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    collate_fn=test_collator,
)


print(
    "\nTest DataLoader samples:",
    len(test_dataset_hf)
)

print(
    "Test inference batch size:",
    TEST_BATCH_SIZE
)

print(
    "\nFinal original test results directory:"
)

print(
    FINAL_ORIGINAL_TEST_DIR
)


# ============================================================
# 10. STORAGE
# ============================================================

all_seed_results = []

all_per_class_results = []

reference_true_labels = None


# ============================================================
# 11. THREE-SEED FINAL TEST LOOP
# ============================================================

for seed in [
    42,
    43,
    44,
]:

    print("\n")
    print("=" * 80)

    print(
        f"ORIGINAL XLM-R FINAL TEST - SEED {seed}"
    )

    print("=" * 80)


    checkpoint_path = (
        FINAL_ORIGINAL_CHECKPOINTS[
            seed
        ]
    )


    # --------------------------------------------------------
    # Cleanup before loading this model
    # --------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    print(
        "Evaluation device:",
        device
    )


    # ========================================================
    # LOAD EXACT VALIDATION-SELECTED CHECKPOINT
    #
    # No forced BF16 model loading.
    # ========================================================

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            checkpoint_path
        )
    )


    model = model.to(
        device
    )

    model.eval()


    print(
        "Loaded checkpoint:"
    )

    print(
        checkpoint_path
    )

    print(
        "Model device:",
        next(
            model.parameters()
        ).device
    )

    print(
        "Model dtype:",
        next(
            model.parameters()
        ).dtype
    )


    # ========================================================
    # DIRECT TEST INFERENCE
    # ========================================================

    predictions = []

    true_labels = []


    with torch.inference_mode():

        for batch in test_loader:

            labels = (
                batch.pop(
                    "labels"
                )
            )


            batch = {
                key: value.to(
                    device
                )
                for key, value
                in batch.items()
            }


            outputs = (
                model(
                    **batch
                )
            )


            batch_predictions = (
                outputs.logits
                .argmax(
                    dim=-1
                )
                .detach()
                .cpu()
                .numpy()
            )


            predictions.extend(
                batch_predictions
            )


            true_labels.extend(
                labels
                .cpu()
                .numpy()
            )


    predictions = np.asarray(
        predictions,
        dtype=int,
    )

    true_labels = np.asarray(
        true_labels,
        dtype=int,
    )


    # ========================================================
    # 12. TEST INTEGRITY CHECKS
    # ========================================================

    assert len(predictions) == 2003
    assert len(true_labels) == 2003


    expected_true_labels = (
        test_df["label"]
        .to_numpy(
            dtype=int
        )
    )


    assert np.array_equal(
        true_labels,
        expected_true_labels,
    ), (
        f"Test label/order mismatch for seed {seed}."
    )


    assert set(
        np.unique(
            predictions
        )
    ).issubset(
        set(
            LABEL_IDS
        )
    )


    if reference_true_labels is None:

        reference_true_labels = (
            true_labels.copy()
        )

    else:

        assert np.array_equal(
            reference_true_labels,
            true_labels,
        )


    print(
        f"\nSeed {seed} test integrity check: PASSED"
    )

    print(
        "Test predictions completed:",
        len(predictions)
    )


    # ========================================================
    # 13. GLOBAL METRICS
    # ========================================================

    accuracy = accuracy_score(
        true_labels,
        predictions,
    )


    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = precision_recall_fscore_support(

        true_labels,
        predictions,

        labels=LABEL_IDS,

        average="macro",

        zero_division=0,
    )


    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _
    ) = precision_recall_fscore_support(

        true_labels,
        predictions,

        labels=LABEL_IDS,

        average="weighted",

        zero_division=0,
    )


    print(
        "\n"
        + "-" * 60
    )

    print(
        f"FINAL TEST RESULTS - SEED {seed}"
    )

    print("-" * 60)


    print(
        f"Accuracy        : {accuracy:.4f}"
    )

    print(
        f"Macro Precision : {macro_precision:.4f}"
    )

    print(
        f"Macro Recall    : {macro_recall:.4f}"
    )

    print(
        f"Macro F1        : {macro_f1:.4f}"
    )

    print(
        f"Weighted F1     : {weighted_f1:.4f}"
    )


    # ========================================================
    # 14. CLASSIFICATION REPORT
    # ========================================================

    report_dict = classification_report(
        true_labels,
        predictions,
        labels=LABEL_IDS,
        target_names=LABEL_NAMES,
        output_dict=True,
        zero_division=0,
    )


    report_df = (
        pd.DataFrame(
            report_dict
        )
        .transpose()
    )


    print(
        f"\nPer-class test results - seed {seed}:"
    )


    display(
        report_df.round(4)
    )


    report_df.to_csv(
        FINAL_ORIGINAL_TEST_DIR
        / (
            f"seed_{seed}_"
            "classification_report.csv"
        )
    )


    # ========================================================
    # 15. SAVE SAMPLE-LEVEL PREDICTIONS
    # ========================================================

    prediction_df = (
        test_df.copy()
    )


    prediction_df[
        "test_index"
    ] = np.arange(
        len(test_df)
    )


    prediction_df[
        "true_label_id"
    ] = true_labels


    prediction_df[
        "predicted_label_id"
    ] = predictions


    prediction_df[
        "true_class"
    ] = [
        id2label[
            int(label)
        ]
        for label
        in true_labels
    ]


    prediction_df[
        "predicted_class"
    ] = [
        id2label[
            int(label)
        ]
        for label
        in predictions
    ]


    prediction_df[
        "correct"
    ] = (
        true_labels
        ==
        predictions
    )


    prediction_df.to_csv(
        FINAL_ORIGINAL_TEST_DIR
        / (
            f"seed_{seed}_"
            "test_predictions.csv"
        ),
        index=False,
        encoding="utf-8",
    )


    # ========================================================
    # 16. CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        true_labels,
        predictions,
        labels=LABEL_IDS,
    )


    assert (
        cm.sum()
        ==
        2003
    )


    cm_df = pd.DataFrame(
        cm,
        index=LABEL_NAMES,
        columns=LABEL_NAMES,
    )


    cm_df.to_csv(
        FINAL_ORIGINAL_TEST_DIR
        / (
            f"seed_{seed}_"
            "confusion_matrix.csv"
        )
    )


    # ========================================================
    # 17. STORE OVERALL SEED RESULT
    # ========================================================

    all_seed_results.append({

        "seed":
            seed,

        "checkpoint":
            str(
                checkpoint_path
            ),

        "test_samples":
            len(
                true_labels
            ),

        "test_accuracy":
            accuracy,

        "test_macro_precision":
            macro_precision,

        "test_macro_recall":
            macro_recall,

        "test_macro_f1":
            macro_f1,

        "test_weighted_precision":
            weighted_precision,

        "test_weighted_recall":
            weighted_recall,

        "test_weighted_f1":
            weighted_f1,
    })


    # ========================================================
    # 18. STORE PER-CLASS RESULTS
    # ========================================================

    for class_id in LABEL_IDS:

        class_name = (
            id2label[
                class_id
            ]
        )


        class_metrics = (
            report_dict[
                class_name
            ]
        )


        all_per_class_results.append({

            "seed":
                seed,

            "class_id":
                class_id,

            "class_name":
                class_name,

            "precision":
                class_metrics[
                    "precision"
                ],

            "recall":
                class_metrics[
                    "recall"
                ],

            "f1":
                class_metrics[
                    "f1-score"
                ],

            "support":
                class_metrics[
                    "support"
                ],
        })


    # ========================================================
    # 19. SAVE INCREMENTALLY
    # ========================================================

    pd.DataFrame(
        all_seed_results
    ).to_csv(
        FINAL_ORIGINAL_TEST_DIR
        / "three_seed_test_results.csv",
        index=False,
    )


    pd.DataFrame(
        all_per_class_results
    ).to_csv(
        FINAL_ORIGINAL_TEST_DIR
        / "three_seed_per_class_raw.csv",
        index=False,
    )


    # ========================================================
    # 20. CLEAN GPU BEFORE NEXT SEED
    # ========================================================

    del outputs
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    print(
        f"\nSeed {seed} final test evaluation safely saved."
    )


# ============================================================
# 21. CROSS-SEED TEST LABEL CONSISTENCY
# ============================================================

assert np.array_equal(
    reference_true_labels,
    test_df[
        "label"
    ]
    .to_numpy(
        dtype=int
    )
)


print(
    "\n"
    + "=" * 70
)

print(
    "CROSS-SEED TEST-LABEL CONSISTENCY: PASSED"
)

print("=" * 70)


# ============================================================
# 22. THREE-SEED OVERALL TEST RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_seed_results
)


assert len(
    results_df
) == 3


print(
    "\n"
    + "=" * 80
)

print(
    "ORIGINAL XLM-R - FINAL THREE-SEED TEST RESULTS"
)

print("=" * 80)


display(
    results_df.round(4)
)


results_df.to_csv(
    FINAL_ORIGINAL_TEST_DIR
    / "three_seed_test_results.csv",
    index=False,
)


# ============================================================
# 23. THREE-SEED MEAN ± SAMPLE STANDARD DEVIATION
# ============================================================

metric_columns = [
    "test_accuracy",
    "test_macro_precision",
    "test_macro_recall",
    "test_macro_f1",
    "test_weighted_f1",
]


summary_df = pd.DataFrame({

    "mean":
        results_df[
            metric_columns
        ]
        .mean(),

    "std":
        results_df[
            metric_columns
        ]
        .std(
            ddof=1
        ),
}).T


print(
    "\nThree-seed final test summary:"
)


display(
    summary_df.round(4)
)


summary_df.to_csv(
    FINAL_ORIGINAL_TEST_DIR
    / "three_seed_test_summary.csv"
)


mean_macro_f1 = (
    results_df[
        "test_macro_f1"
    ]
    .mean()
)


std_macro_f1 = (
    results_df[
        "test_macro_f1"
    ]
    .std(
        ddof=1
    )
)


print(
    "\nFINAL TEST MACRO F1:"
)

print(
    f"{mean_macro_f1:.4f}"
    " ± "
    f"{std_macro_f1:.4f}"
)


# ============================================================
# 24. THREE-SEED PER-CLASS SUMMARY
# ============================================================

per_class_df = pd.DataFrame(
    all_per_class_results
)


assert (
    len(per_class_df)
    ==
    15
)


per_class_summary = (

    per_class_df

    .groupby(
        [
            "class_id",
            "class_name",
        ],
        as_index=False,
    )

    .agg(

        precision_mean=(
            "precision",
            "mean"
        ),

        precision_std=(
            "precision",
            "std"
        ),

        recall_mean=(
            "recall",
            "mean"
        ),

        recall_std=(
            "recall",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),

        f1_std=(
            "f1",
            "std"
        ),

        support=(
            "support",
            "first"
        ),
    )
)


print(
    "\nThree-seed per-class FINAL TEST summary:"
)


display(
    per_class_summary.round(4)
)


per_class_summary.to_csv(
    FINAL_ORIGINAL_TEST_DIR
    / "three_seed_per_class_summary.csv",
    index=False,
)


# ============================================================
# 25. SAVE METADATA
# ============================================================

metadata = {

    "experiment":
        "Original RUHSOLD XLM-R matched 3-seed baseline",

    "model":
        MODEL_NAME,

    "train_samples":
        6408,

    "validation_samples":
        801,

    "test_samples":
        2003,

    "seeds":
        [
            42,
            43,
            44,
        ],

    "test_class_distribution":
        {
            str(k): int(v)
            for k, v
            in actual_test_counts.items()
        },

    "checkpoints":
        {
            str(seed):
                str(
                    FINAL_ORIGINAL_CHECKPOINTS[
                        seed
                    ]
                )
            for seed
            in [
                42,
                43,
                44,
            ]
        },

    "checkpoint_selection":
        (
            "Best checkpoint selected using "
            "validation Macro-F1 only."
        ),

    "test_set_usage":
        (
            "Final evaluation only. "
            "No tuning or checkpoint selection "
            "performed using the test set."
        ),

    "mean_test_macro_f1":
        float(
            mean_macro_f1
        ),

    "std_test_macro_f1":
        float(
            std_macro_f1
        ),
}


with open(
    FINAL_ORIGINAL_TEST_DIR
    / "final_test_metadata.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
    )


# ============================================================
# 26. FINAL VERIFICATION
# ============================================================

assert (
    results_df[
        "test_samples"
    ]
    == 2003
).all()


assert (
    results_df[
        "seed"
    ]
    .tolist()
    ==
    [
        42,
        43,
        44,
    ]
)


print(
    "\n"
    + "=" * 80
)

print(
    "ORIGINAL XLM-R FINAL TEST EVALUATION COMPLETE"
)

print("=" * 80)


print(
    "Seeds evaluated:",
    [
        42,
        43,
        44,
    ]
)


print(
    "Test samples per seed:",
    results_df[
        "test_samples"
    ].tolist()
)


print(
    "\nMean test Macro F1:",
    f"{mean_macro_f1:.4f}"
)


print(
    "Test Macro F1 standard deviation:",
    f"{std_macro_f1:.4f}"
)


print(
    "\nAll final original XLM-R test results saved to:"
)

print(
    FINAL_ORIGINAL_TEST_DIR
)

ORIGINAL XLM-R FINAL TEST CHECKPOINTS
Seed 42: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/diagnostic_seed_42/checkpoint-2005
Seed 43: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/seed_43/checkpoint-2005
Seed 44: True /home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed/seed_44/checkpoint-1203

All three original XLM-R checkpoints verified.

FINAL TEST-SET INTEGRITY CHECK
Test samples: 2003
Missing tweets: 0
Missing labels: 0

Test class distribution:


,label,count
0,0,481
1,1,1070
2,2,156
3,3,168
4,4,128



Test-set integrity check: PASSED


Map:   0%|          | 0/2003 [00:00<?, ? examples/s]


Test DataLoader samples: 2003
Test inference batch size: 4

Final original test results directory:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed_results/final_test


ORIGINAL XLM-R FINAL TEST - SEED 42
Evaluation device: cuda


...


Three-seed final test summary:


,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1
mean,0.7943,0.7156,0.7080,0.7078,0.7931
std,0.0058,0.0068,0.0112,0.0113,0.0071



FINAL TEST MACRO F1:
0.7078 ± 0.0113

Three-seed per-class FINAL TEST summary:


,class_id,class_name,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,support
0,0,Abusive/Offensive,0.7202,0.0181,0.6667,0.0526,0.6911,0.0212,481.0
1,1,Normal,0.8887,0.0130,0.9100,0.0186,0.8990,0.0030,1070.0
2,2,Religious Hate,0.7069,0.0694,0.6538,0.0611,0.6754,0.0169,156.0
3,3,Sexism,0.6311,0.0926,0.6766,0.0821,0.6453,0.0144,168.0
4,4,Profane,0.6311,0.0813,0.6328,0.0435,0.6282,0.0216,128.0



ORIGINAL XLM-R FINAL TEST EVALUATION COMPLETE
Seeds evaluated: [42, 43, 44]
Test samples per seed: [2003, 2003, 2003]

Mean test Macro F1: 0.7078
Test Macro F1 standard deviation: 0.0113

All final original XLM-R test results saved to:
/home/jovyan/project work/data_analyssis/outputs/xlm_roberta_original_final_3seed_results/final_test
